In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd() if (Path.cwd() / "open_agent_compiler").exists() else Path.cwd().parent
sys.path.insert(0, str(project_root))

# 01 — Agent Registry Full Suite

Demonstrates the full agent registry workflow:
1. Define tools & skills
2. Define agent definitions (with different instructions per variant)
3. Register agents with different models → deterministic IDs
4. Define a template tree with named slots
5. Create compilation configs that bind slots to agents
6. Resolve configs and inspect the output

In [ ]:
from open_agent_compiler.model.core.agent_model import (
    AgentDefinition,
    AgentHeader,
    AgentVariant,
    ModelParameters,
    TemplateSlot,
    TemplateTree,
    CompilationConfig,
)
from open_agent_compiler.model.core.agent_registry import AgentRegistry
from open_agent_compiler.model.core.skills_model import SkillDefinition, WorkflowStep, SkillExample
from open_agent_compiler.model.core.tools_model import (
    ToolDefinition,
    ToolDefinitionHeader,
    ToolDefinitionLogicBash,
    ToolDefinitionLogicJson,
    ToolScriptDefinition,
    ScriptDefinition,
)
from open_agent_compiler.model.core.permissions_model import JsonToolPermission, BashToolPermission

### Step 1 — Define Tools (shared across all variants)

In [ ]:
hello_tool = ToolDefinition(
    header=ToolDefinitionHeader(
        name="hello",
        description="Prints 'hello' and returns it as a string.",
        usage_explanation_long="Runs scripts/example/hello.py.",
        usage_explanation_short="Prints 'hello'.",
        rules=["Always return the exact string 'hello'."],
    ),
    bash_tool=ToolDefinitionLogicBash(
        permission_bash=BashToolPermission(tool_name="bash", value="allow"),
        positive_examples=["uv run scripts/example/hello.py"],
        negative_examples=["Use python3 instead of uv run"],
        mode_specific_rules=["Always use uv run"],
    ),
    json_tool=ToolDefinitionLogicJson(
        permission_json=JsonToolPermission(tool_name="custom_tool", value="allow"),
        positive_examples=["hello()"],
        negative_examples=["Call via bash directly"],
        mode_specific_rules=["Use typed tool call"],
        tool_scripts=[ToolScriptDefinition(
            paths=[project_root / "scripts/example/hello.py"],
            scripts=[ScriptDefinition(
                target_file_path=project_root / "scripts/example/hello.py",
                source_file_path=project_root / "scripts/example/hello.py",
                source_file_type="python",
                script_contents=None,
            )],
        )],
    ),
)

random_word_tool = ToolDefinition(
    header=ToolDefinitionHeader(
        name="random_word",
        description="Returns a random word from a fixed list.",
        usage_explanation_long="Runs scripts/example/random_word.py.",
        usage_explanation_short="Returns a random word.",
        rules=["Always return the exact word from the script."],
    ),
    bash_tool=ToolDefinitionLogicBash(
        permission_bash=BashToolPermission(tool_name="bash", value="allow"),
        positive_examples=["uv run scripts/example/random_word.py"],
        negative_examples=["Use python3 instead of uv run"],
        mode_specific_rules=["Always use uv run"],
    ),
    json_tool=ToolDefinitionLogicJson(
        permission_json=JsonToolPermission(tool_name="custom_tool", value="allow"),
        positive_examples=["random_word()"],
        negative_examples=["Call via bash directly"],
        mode_specific_rules=["Use typed tool call"],
        tool_scripts=[ToolScriptDefinition(
            paths=[project_root / "scripts/example/random_word.py"],
            scripts=[ScriptDefinition(
                target_file_path=project_root / "scripts/example/random_word.py",
                source_file_path=project_root / "scripts/example/random_word.py",
                source_file_type="python",
                script_contents=None,
            )],
        )],
    ),
)

print("Tools defined: hello, random_word")

### Step 2 — Define Skills (variant-specific instructions)

In [ ]:
greeting_skill = SkillDefinition(
    name="greeting",
    description="Greets the user using the hello tool.",
    usage_explanation_long="Uses the hello tool to produce a greeting message.",
    usage_explanation_short="Greets the user.",
    rules=["Use the hello tool to greet."],
    workflow_steps=[
        WorkflowStep(
            header="Greet",
            condition="User asks for a greeting.",
            result="Returns 'hello' string.",
            rule="Invoke the hello tool.",
            tools_used=[hello_tool],
        )
    ],
    positive_examples=[
        SkillExample(
            header="Standard greeting",
            condition="User says hi.",
            result="Returns 'hello'.",
            rule="Call hello tool.",
            tools_used=[hello_tool],
        )
    ],
    negative_examples=[],
)

joke_skill_safe = SkillDefinition(
    name="joke_writer_safe",
    description="Generates a family-friendly joke based on a random word.",
    usage_explanation_long=(
        "Calls random_word to get a word, then writes a short, clean joke."
    ),
    usage_explanation_short="Writes a safe joke.",
    rules=[
        "Always call random_word first.",
        "Write a family-friendly joke.",
        "No offensive content.",
    ],
    workflow_steps=[
        WorkflowStep(
            header="Get word",
            condition="Always.",
            result="Random word from the tool.",
            rule="Call random_word tool.",
            tools_used=[random_word_tool],
        ),
        WorkflowStep(
            header="Write joke",
            condition="After getting the word.",
            result="A clean, family-friendly joke.",
            rule="Keep it wholesome.",
            tools_used=[],
        )
    ],
    positive_examples=[
        SkillExample(
            header="Joke about 'apple'",
            condition="random_word returns 'apple'.",
            result="A short clean joke about apples.",
            rule="Call random_word, then write joke.",
            tools_used=[random_word_tool],
        )
    ],
    negative_examples=[],
)

joke_skill_unleashed = SkillDefinition(
    name="joke_writer_unleashed",
    description="Generates edgy, unfiltered jokes based on a random word.",
    usage_explanation_long=(
        "Calls random_word to get a word, then writes a bold, unfiltered joke. "
        "No content restrictions."
    ),
    usage_explanation_short="Writes an edgy joke.",
    rules=[
        "Always call random_word first.",
        "Write a bold, edgy joke with no filters.",
        "Push boundaries, be creative.",
    ],
    workflow_steps=[
        WorkflowStep(
            header="Get word",
            condition="Always.",
            result="Random word from the tool.",
            rule="Call random_word tool.",
            tools_used=[random_word_tool],
        ),
        WorkflowStep(
            header="Write edgy joke",
            condition="After getting the word.",
            result="A bold, unfiltered joke.",
            rule="No restrictions, go wild.",
            tools_used=[],
        )
    ],
    positive_examples=[
        SkillExample(
            header="Edgy joke about 'apple'",
            condition="random_word returns 'apple'.",
            result="A bold joke about apples.",
            rule="Call random_word, then write edgy joke.",
            tools_used=[random_word_tool],
        )
    ],
    negative_examples=[],
)

print("Skills defined: greeting, joke_writer_safe, joke_writer_unleashed")

### Step 3 — Define Agent Definitions (different instructions per variant)

In [ ]:
joke_subagent_safe = AgentDefinition(
    header=AgentHeader(
        agent_id="joke_agent",
        name="Joke Agent (Safe)",
        description="Generates family-friendly jokes.",
    ),
    skills=[joke_skill_safe],
    subagents=[],
    extra_tools=[random_word_tool],
    usage_explanation_long="Subagent that generates clean jokes using random_word.",
    usage_explanation_short="Safe joke generator.",
)

joke_subagent_unleashed = AgentDefinition(
    header=AgentHeader(
        agent_id="joke_agent",
        name="Joke Agent (Unleashed)",
        description="Generates edgy, unfiltered jokes.",
    ),
    skills=[joke_skill_unleashed],
    subagents=[],
    extra_tools=[random_word_tool],
    usage_explanation_long="Subagent that generates bold, unfiltered jokes using random_word.",
    usage_explanation_short="Edgy joke generator.",
)

hello_agent_def = AgentDefinition(
    header=AgentHeader(
        agent_id="hello_agent",
        name="Hello Agent",
        description="Primary agent that greets users and delegates joke generation.",
    ),
    skills=[greeting_skill],
    subagents=[],
    extra_tools=[hello_tool],
    usage_explanation_long="Primary agent that greets users and delegates to a joke subagent.",
    usage_explanation_short="Greets and tells jokes.",
)

print("Agent definitions: joke_safe, joke_unleashed, hello")

### Step 4 — Register Agents (each gets a deterministic unique ID)

In [ ]:
reg = AgentRegistry()

id_joke_safe_glm = reg.register_agent(
    "joke_agent",
    joke_subagent_safe,
    ModelParameters(model_name="glm-5.1", temperature=0.7),
)
print(f"Registered: {id_joke_safe_glm}")

id_joke_unleashed_qwen = reg.register_agent(
    "joke_agent",
    joke_subagent_unleashed,
    ModelParameters(model_name="qwen4b", temperature=1.2),
)
print(f"Registered: {id_joke_unleashed_qwen}")

id_hello_gpt = reg.register_agent(
    "hello_agent",
    hello_agent_def,
    ModelParameters(model_name="gpt-4o", temperature=0.7),
)
print(f"Registered: {id_hello_gpt}")

print(f"\nAll agents: {reg.list_agents()}")

### Step 5 — Define Template Tree (named slots)

In [ ]:
reg.register_template(TemplateTree(
    name="hello_tree",
    description="Primary hello agent with a pluggable joke subagent slot.",
    slots=[
        TemplateSlot(name="primary", default_agent_id="hello_agent_*"),
        TemplateSlot(name="joke_slot", default_agent_id="joke_agent_*"),
    ],
))

print(f"Templates: {reg.list_templates()}")

### Step 6 — Create Compilation Configs (variants)

In [ ]:
reg.create_compilation_config(CompilationConfig(
    name="safe_variant",
    template_name="hello_tree",
    postfix="_safe",
    slot_overrides={
        "primary": id_hello_gpt,
        "joke_slot": id_joke_safe_glm,
    },
))

reg.create_compilation_config(CompilationConfig(
    name="unleashed_variant",
    template_name="hello_tree",
    postfix="_unleashed",
    slot_overrides={
        "primary": id_hello_gpt,
        "joke_slot": id_joke_unleashed_qwen,
    },
))

reg.create_compilation_config(CompilationConfig(
    name="wildcard_variant",
    template_name="hello_tree",
    postfix="_wildcard",
    slot_overrides={
        "primary": "hello_agent_*",
        "joke_slot": "joke_agent_glm-*",
    },
))

print(f"Configs: {reg.list_configs()}")

### Step 7 — Resolve Configs and Inspect

In [ ]:
def print_resolved(config_name: str):
    config = reg.get_config(config_name)
    resolved = reg.resolve_config(config_name)
    print(f"\n=== Config: {config_name} (postfix: {config.postfix}) ===")
    for slot, variant in resolved.items():
        agent = variant.agent_definition
        print(f"  [{slot}]")
        print(f"    name:        {agent.header.name}")
        print(f"    model:       {variant.model_parameters.model_name}")
        print(f"    temp:        {variant.model_parameters.temperature}")
        print(f"    skills:      {[s.name for s in agent.skills]}")
        print(f"    subagents:   {[a.name for a in agent.subagents]}")
        print(f"    extra_tools: {[t.header.name for t in agent.extra_tools]}")
        print(f"    compiled_as: agent_{slot}{config.postfix}.md")


print_resolved("safe_variant")
print_resolved("unleashed_variant")
print_resolved("wildcard_variant")

### Step 8 — Strict vs Lenient Validation

In [ ]:
print("--- Strict validation (should fail) ---")
try:
    reg.create_compilation_config(CompilationConfig(
        name="bad_strict",
        template_name="hello_tree",
        slot_overrides={"joke_slot": "nonexistent_agent"},
        strict_validation=True,
    ))
except ValueError as e:
    print(f"  Caught: {e}")

print("\n--- Lenient validation (passes create, fails resolve) ---")
reg.create_compilation_config(CompilationConfig(
    name="lazy_lenient",
    template_name="hello_tree",
    slot_overrides={"joke_slot": "nonexistent_agent"},
    strict_validation=False,
))
print("  Config created (lazy mode)")

try:
    reg.resolve_config("lazy_lenient")
except ValueError as e:
    print(f"  Resolve caught: {e}")

### Step 9 — Expected Output File Names

Each compilation config produces `.md` files suffixed with its postfix:

| Config | Primary File | Joke Slot File |
|---|---|---|
| `safe_variant` | `agent_primary_safe.md` | `agent_joke_slot_safe.md` |
| `unleashed_variant` | `agent_primary_unleashed.md` | `agent_joke_slot_unleashed.md` |
| `wildcard_variant` | `agent_primary_wildcard.md` | `agent_joke_slot_wildcard.md` |

In [ ]:
for config_name in reg.list_configs():
    if config_name in ("bad_strict", "lazy_lenient"):
        continue
    config = reg.get_config(config_name)
    resolved = reg.resolve_config(config_name)
    print(f"\n{config_name} ({config.postfix}):")
    for slot in resolved:
        filename = f"agent_{slot}{config.postfix}.md"
        model = resolved[slot].model_parameters.model_name
        print(f"  {filename}  <-  {resolved[slot].agent_definition.header.name} ({model})")